# Micrograd: A Scalar Autodiff Engine — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

`loss.backward()` is **reverse-mode automatic differentiation**. During the forward pass every operation records its inputs and how to push a gradient back through itself; `backward()` then walks the recorded graph in *reverse topological order*, applying the chain rule once per edge. Karpathy's *micrograd* shows the whole idea fits in ~100 lines of Python on scalars, and "implement autograd for scalars" is a classic research-engineer interview question because it tests the chain rule, graph traversal and Python operator overloading at once. PyTorch's autograd engine is the same algorithm with tensors as nodes and vector-Jacobian products as the local backward functions.

Build a `Value` class that behaves like a scalar `torch.tensor(x, dtype=torch.float64, requires_grad=True)`.

---

### Requirements

1. **`Value(data)`** — attributes `.data` (a Python `float`) and `.grad` (a Python `float`, starting at `0.0`).
2. **Arithmetic** producing new `Value`s: `a + b`, `a - b`, `a * b`, `a / b` and unary `-a`, where the other operand may be a `Value` **or a Python int/float on either side**: `v + 1`, `2 * v`, `1 - v`, `1 / v`, `3.0 + v` must all work (`__radd__`, `__rsub__`, `__rmul__`, `__rtruediv__`).
3. **Power by a constant**: `a ** k` for an int/float `k`, including negative and fractional exponents (e.g. `a ** -1.5` for `a > 0`). The exponent is never a `Value`.
4. **Functions**: `.exp()`, `.log()` (natural log), `.tanh()`, `.relu()`, `.sigmoid()`. `relu` has gradient **0 at exactly 0** (PyTorch's convention).
5. **`out.backward()`** sets `out.grad = 1.0` and propagates to every node `out` depends on, calling each node's local backward only after **all** of its consumers have contributed to its `.grad` (reverse topological order).
6. **Gradients accumulate**: a node used more than once (`x * x`, or `a = x + y; a * a`) receives the **sum** of the contributions from every use.
7. **No recursion limit**: `backward()` must work on a chain 5000 nodes deep at Python's default recursion limit (1000) — build the topological order with an explicit stack.
8. `backward()` is called once per freshly built graph (like PyTorch without `retain_graph=True`); you don't need to support calling it twice or zeroing gradients.

---

### Constraints

- ✅ Pure Python and the `math` module.
- ❌ No torch / numpy inside `Value` (torch is only used by the tests as the oracle).
- ❌ Don't make a recursive topological sort pass by raising `sys.setrecursionlimit` — use an iterative traversal.

---

<details>
  <summary>💡 Hint</summary>

  - Each operation computes `out.data`, remembers its inputs, and attaches a closure `out._backward` that knows the *local* derivatives and the chain rule: input.grad (something) local_derivative × out.grad. Should "(something)" be `=` or `+=`?
  - A topological order lists every node after all of its inputs; a post-order depth-first search produces one. To avoid recursion, push `(node, children_done)` pairs onto a list used as a stack.
  - You only need a few real primitives — `+`, `*`, `** k`, `exp`, `log`, `tanh`, `relu`, `sigmoid`. Express `-a`, `a - b` and `a / b` through them.
  - For `1 - v`, Python first tries `int.__sub__(1, v)`, which returns `NotImplemented`, and then calls `v.__rsub__(1)`. Mind the operand order.
  - Express derivatives through the *output* where that is cheaper: $\tanh' = 1 - \tanh^2$, $\sigma' = \sigma(1 - \sigma)$, $\exp' = \exp$.

</details>

---

In [ ]:
import math
import random
import sys

import torch   # the tests use torch.autograd as the oracle; Value itself must not use torch

In [ ]:
def torch_scalar(x):
    """A float64 leaf tensor that plays the role of Value(x) in the tests."""
    return torch.tensor(float(x), dtype=torch.float64, requires_grad=True)


print("default recursion limit:", sys.getrecursionlimit())

In [ ]:
class Value:
    """A scalar node in a computation graph, with reverse-mode autodiff."""

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = tuple(_children)     # the inputs this node was computed from
        self._op = _op                    # for debugging / printing only
        self._backward = lambda: None     # leaves have nothing to propagate to

    # ---- primitives -----------------------------------------------------------
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad         # +=, never =: a node may feed several consumers
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents are supported"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def _unary(self, value, local_grad, op):
        """out = f(self) with the local derivative f'(self) precomputed."""
        out = Value(value, (self,), op)

        def _backward():
            self.grad += local_grad * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        return self._unary(e, e, "exp")                       # d/dx e^x = e^x

    def log(self):
        return self._unary(math.log(self.data), 1.0 / self.data, "log")

    def tanh(self):
        t = math.tanh(self.data)
        return self._unary(t, 1.0 - t * t, "tanh")            # uses the OUTPUT

    def relu(self):
        return self._unary(max(self.data, 0.0), float(self.data > 0), "relu")

    def sigmoid(self):
        x = self.data                                         # stable in both tails
        s = 1.0 / (1.0 + math.exp(-x)) if x >= 0 else math.exp(x) / (1.0 + math.exp(x))
        return self._unary(s, s * (1.0 - s), "sigmoid")

    # ---- derived ops ----------------------------------------------------------
    def __neg__(self):              return self * -1
    def __sub__(self, other):       return self + (-other)
    def __truediv__(self, other):   return self * other ** -1
    def __radd__(self, other):      return self + other            # other + self
    def __rmul__(self, other):      return self * other            # other * self
    def __rsub__(self, other):      return Value(other) - self     # other - self (order matters)
    def __rtruediv__(self, other):  return Value(other) / self     # other / self

    # ---- reverse-mode sweep ---------------------------------------------------
    def backward(self):
        # Iterative post-order DFS: a node is appended only after all of its inputs,
        # so `topo` is a topological order. No recursion, hence no depth limit.
        topo, visited = [], set()
        stack = [(self, False)]
        while stack:
            node, children_done = stack.pop()
            if children_done:
                topo.append(node)
                continue
            if node in visited:
                continue
            visited.add(node)
            stack.append((node, True))                        # revisit after the children
            for child in node._prev:
                if child not in visited:
                    stack.append((child, False))
        self.grad = 1.0
        for node in reversed(topo):                           # consumers before their inputs
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data:.6g}, grad={self.grad:.6g})"

In [ ]:
# Test 1 — a small expression differentiated by hand
a, b = Value(2.0), Value(-3.0)
c = a * b + b ** 2              # -6 + 9 = 3
c.backward()
assert c.data == 3.0, c
assert a.grad == -3.0, f"dc/da = b = -3, got {a.grad}"
assert b.grad == -4.0, f"dc/db = a + 2b = -4, got {b.grad}"
print("hand-derived gradients OK:", a, b)

In [ ]:
# Test 2 — Python numbers on either side of an operator
for expr, f in [("2 * v", lambda v: 2 * v), ("3 + v", lambda v: 3 + v), ("1 - v", lambda v: 1 - v),
                ("1 / v", lambda v: 1 / v), ("v / 4", lambda v: v / 4), ("v - 2", lambda v: v - 2),
                ("v ** -1.5", lambda v: v ** -1.5), ("-v", lambda v: -v)]:
    v, t = Value(4.0), torch_scalar(4.0)
    out_v, out_t = f(v), f(t)
    out_v.backward()
    out_t.backward()
    assert abs(out_v.data - out_t.item()) < 1e-12, f"{expr}: value {out_v.data} vs {out_t.item()}"
    assert abs(v.grad - t.grad.item()) < 1e-12, f"{expr}: grad {v.grad} vs {t.grad.item()}"
print("reflected ops OK")

In [ ]:
# Test 3 — a node used several times must receive the SUM of its gradient contributions
x = Value(3.0)
y = x * x
y.backward()
assert x.grad == 6.0, f"d(x*x)/dx = 2x = 6, got {x.grad}"

x, y = Value(1.5), Value(-0.5)
a = x + y
out = a * a                     # = (x + y)^2
out.backward()
assert x.grad == 2.0 and y.grad == 2.0, f"expected 2(x+y) = 2 for both, got {x.grad}, {y.grad}"
print("shared nodes accumulate OK")

In [ ]:
# Test 4 — every op in one expression with shared subexpressions, vs torch.autograd
def expression(x, y, z):
    a = x * y + z.exp()               # a is used three times below
    b = (a / (1 + y ** 2)).tanh()
    c = (z ** 2 + 0.5).log() - a.sigmoid() * 2
    d = (b - c).relu() + (1 - x) / 3
    return d * a + b ** -2 + (-c) * a


inputs = (0.6, -1.2, 0.3)
vals = [Value(v) for v in inputs]
tens = [torch_scalar(v) for v in inputs]
out_v, out_t = expression(*vals), expression(*tens)
out_v.backward()
out_t.backward()
print(f"output: Value {out_v.data:.10f}   torch {out_t.item():.10f}")
assert abs(out_v.data - out_t.item()) < 1e-9
for name, v, t in zip("xyz", vals, tens):
    print(f"d/d{name}: Value {v.grad:+.10f}   torch {t.grad.item():+.10f}")
    assert abs(v.grad - t.grad.item()) < 1e-9, f"wrong gradient for {name}"

In [ ]:
# Test 5 — a 5000-deep chain at Python's default recursion limit
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    x = Value(0.5)
    y = x
    for _ in range(5000):
        y = y + x                     # y = 5001 * x
    y.backward()
finally:
    sys.setrecursionlimit(old_limit)
assert x.grad == 5001.0, f"expected 5001, got {x.grad}"
print("deep chain OK:", x)

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("micrograd", Value=Value)

### Demo: a tiny MLP trained with your engine

Neurons, layers and an MLP are just bookkeeping on top of `Value`. The toy task is XOR-like (label $= \operatorname{sign}(x_1 x_2)$), which no linear model can
solve. Note the `p.grad = 0.0` before every `backward()`: the parameters are leaves that are reused by every new graph, and `backward` *accumulates* —
exactly why PyTorch needs `optimizer.zero_grad()`.

In [ ]:
class Neuron:
    def __init__(self, n_in, nonlin=True):
        self.w = [Value(random.uniform(-1, 1) / math.sqrt(n_in)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out, nonlin=True):
        self.neurons = [Neuron(n_in, nonlin) for _ in range(n_out)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class TinyMLP:
    def __init__(self, n_in, sizes):
        dims = [n_in] + sizes
        self.layers = [Layer(dims[i], dims[i + 1], nonlin=i < len(sizes) - 1) for i in range(len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


random.seed(1)
X_toy = []
while len(X_toy) < 40:                                  # keep a small margin around the axes
    p = (random.uniform(-1, 1), random.uniform(-1, 1))
    if abs(p[0]) > 0.15 and abs(p[1]) > 0.15:
        X_toy.append(p)
Y_toy = [1.0 if a * b > 0 else -1.0 for a, b in X_toy]

model = TinyMLP(2, [8, 1])
print(len(model.parameters()), "parameters")
for step in range(60):
    scores = [model(x) for x in X_toy]
    loss = sum(((1 - yi * si).relu() for yi, si in zip(Y_toy, scores)), Value(0.0)) / len(X_toy)  # hinge
    for p in model.parameters():
        p.grad = 0.0                                    # leaves accumulate across graphs!
    loss.backward()
    for p in model.parameters():
        p.data -= 1.0 * p.grad                          # SGD, lr = 1.0
    if step % 10 == 0 or step == 59:
        acc = sum((si.data > 0) == (yi > 0) for yi, si in zip(Y_toy, scores)) / len(X_toy)
        print(f"step {step:2d}   hinge loss {loss.data:.4f}   accuracy {acc:.0%}")

## Why it works

For a scalar output $L$ and any node $u$, the multivariate chain rule says

$$\bar u \;\equiv\; \frac{\partial L}{\partial u} \;=\; \sum_{v\,\in\,\text{consumers}(u)} \bar v\, \frac{\partial v}{\partial u}.$$

Each `_backward` closure adds one term of this sum ($\bar v \cdot \partial v/\partial u$) into `u.grad`, so `+=` *is* the sum over consumers. Visiting nodes in
reverse topological order guarantees that when a node's `_backward` runs, its own $\bar v$ is already final (every consumer has run). Each edge is processed
exactly once, so a backward pass costs a small constant times the forward pass, *independently of how many inputs there are* — the reason reverse mode
(a.k.a. backprop) is the right choice for one scalar loss and millions of parameters.

Local derivatives used: $\partial(a+b) = (1, 1)$, $\partial(ab) = (b, a)$, $(x^k)' = kx^{k-1}$, $(e^x)' = e^x$, $(\ln x)' = 1/x$, $\tanh' = 1 - \tanh^2$,
$\sigma' = \sigma(1-\sigma)$, $\mathrm{relu}' = \mathbb{1}[x > 0]$. Subtraction and division reuse them: $a - b = a + (-1)b$, $a / b = a \cdot b^{-1}$
(so $\partial_b = -a/b^2$ comes for free).

The iterative DFS pushes `(node, False)` to expand a node and `(node, True)` to emit it after its children; in a DAG a node can't be "in progress" when one of
its consumers is emitted (that would be a cycle), so the emitted order is topological.

## Complexity

Topological sort and backward are $O(V + E)$ time; memory is $O(V)$ because every intermediate `Value` (and its closure) stays alive until backward — the
scalar analogue of activation memory. Scalar graphs are ~$10^5\times$ slower than tensor autograd because of Python overhead per node; real engines put a whole
tensor in each node and make `_backward` a vector-Jacobian product (e.g. for a matmul: $\bar A = \bar C B^\top$, $\bar B = A^\top \bar C$).

## Common bugs

- `self.grad = ...` instead of `+=` — right for trees, wrong as soon as a node is reused (`x * x` gives $x$ instead of $2x$).
- A recursive `build_topo` (as in the original micrograd) → `RecursionError` on deep graphs (long RNN unrolls, deep chains).
- Calling `_backward` in DFS or BFS order instead of reverse topological order: a shared node propagates before all its consumers have contributed.
- Reflected ops with the operands swapped: `__rsub__` returning `self - other` makes `1 - v` equal `v - 1`; same for `__rtruediv__`.
- $\tanh'$ written as $1 - x^2$ (the input) instead of $1 - t^2$ (the output); $(x^k)' = kx^k$.
- Forgetting to wrap Python numbers (`other.data` on an `int`), or capturing a loop variable in a closure late-binding style.
- Not zeroing parameter grads between training steps — parameters are leaves that survive across graphs.
- Defining `__eq__` for elementwise comparison without `__hash__` makes `Value` unhashable, breaking `visited` sets (use `id(node)` then).

## Interview follow-ups

1. **Forward mode vs reverse mode?** Forward mode (dual numbers) pushes a tangent alongside each value: one pass per *input* direction gives a Jacobian-vector product — good for few inputs/many outputs. Reverse mode gives a vector-Jacobian product per *output* — good for a scalar loss. JAX exposes both as `jvp` / `vjp`.
2. **How does PyTorch's engine differ?** Nodes are tensors and each op records a `grad_fn` with saved tensors; the engine schedules nodes by dependency counts (Kahn's algorithm) instead of an explicit topo sort, runs per-device threads, accumulates into `.grad` only for leaves (`retain_grad()` for others), and frees saved tensors after backward unless `retain_graph=True`.
3. **Higher-order derivatives?** Make the backward pass itself build `Value`s (record the ops it performs) — PyTorch's `create_graph=True` — then call `backward` on a gradient, e.g. for Hessian-vector products, MAML, or gradient penalties (WGAN-GP).
4. **Why do in-place ops break autograd?** `_backward` closures read saved values (`other.data`, the output `t`) at backward time; overwriting them in place silently corrupts gradients. PyTorch tracks a version counter on saved tensors and raises an error instead.
5. **What would you need to vectorize this?** Tensor-valued nodes, VJP rules per op, and "unbroadcasting": a gradient flowing into a broadcast input must be summed back to that input's shape (the same reason `grad_b` is a sum over the batch).
6. **Why does `relu'(0) = 0` matter?** ReLU isn't differentiable at 0; any value in $[0, 1]$ is a valid subgradient. Matching PyTorch's choice makes gradient tests deterministic, and it matters in practice for exactly-zero pre-activations (zero-initialised inputs/biases, quantized or padded inputs).